Cargamos librerías y definimos constantes que nos servirán en la descarga de datos de la AEMET

incluyendo:
- la api key
- la url base de la AEMET
- headers : que solo llevarán la api key (en lugar de ponerlos en parámetros)


In [ ]:
import os
import requests
from requests.exceptions import HTTPError, RequestException 
import json
import pandas as pd
import time


pd.set_option("display.max_colwidth", None) # Claude Opus 5.5 (bajo)

API_KEY  = os.environ["AEMET_API_KEY"] # guardada como variable de entorno en el terminal
BASE_URL = "https://opendata.aemet.es/opendata/api"
HEADERS = {"api_key" : API_KEY}

Vamos a definir dos funciones que utilizaremos constantemente en el proceso de la adquisición de datos de la AEMET:

- **llamada control**: 
    - La API de la AEMET no devuelve los datos en una llamada: devuelve un diccionario con la información de los datos y dónde conseguirlos
    - Podemos esperar un error HTTP o un error dentro de la API: comprobaremos su estado
    - Tiene una caducidad de 5 minutos según la propia documentación de la AEMET
    - utiliza nuestra API KEY, la url base y el endpoint al que deseamos acceder
    - convertimos el JSON que proporciona a un diccionario Python utilizando `.json()`

- **llamada general**:
    - utilizamos la url que nos proporciona la llamada de control
    - no requiere de nuestra API KEY ya que la url actúa de sustituta
    - siempre debe ir emparejada con la llamada de control por la caducidad de la URL
    - tiene en cuenta que la AEMET no siempre proprociona los datos codificados en UTF-8: prueba también con latin-1
    - guarda los datos en una lista de diccionarios con `json.loads()`

- **llamda lanzadera/datos**:
    - llama a las otras funciones en orden para mantenerlas siempre juntas
    - necesitará un endpoint para la llamada de control
    - devuelve de manera opcional los metadatos que acompañan a los datos pedidos

- **Datos de LLuvia**:
    - cogeremos los datos de las estaciones en la provincia de Valencia (que posean datos meteorológicos) utilizando los idema extraídos anteriormente
    - datos de las últimas 12 horas
    - el endpoint contiene la ruta y el parámetro idema (identificador de la estación)
    - usamos `.extend()`en lugar de `.append()` para aplanar 
    - datos_lluvia es una lista de diccionarios donde cada diccionario corresponde a la medición en 1 hora de una estación particular. 
    - Hay estaciones que pueden no ofrecer datos en vivo (UPV, VIVEROS), estas se *guardan en una lista con su motivo* 
    - Esta nueva versión recoge:
        -  los errores en el nivel 1 del código (HTTP) en el caso de que haya un exceso de peticiones, o que no se haya encontrado. En el caso de ser un error 429, esperará e intentará acceder a la misma estación al continuar, si es un error 404 simplemente apunta la estación y el error que le ha dado.
        - los errores en el nivel 2 del código (JSON): al comprobar el estado del JSON podemos encontrarnos un RuntimeError, como esto es un error del estado de la API lo que hacemos es apuntarlo y pasar a la siguiente estación. 
        - Errores del propio requests: timeout/sin conexión, esperamos y reeintentamos.
        - cada reintento usa una llamada_datos nueva para evitar que caduque la URL. 


In [ ]:
def llamada_control(endpoint):

    r = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, timeout=10)
    r.raise_for_status() # nivel 1: código HTTP

    control = r.json()

    estado = control.get("estado")
    if estado != 200: # nivel 2: estado de la API
        raise RuntimeError(estado, control.get("descripcion"))
    
    return control


def llamada_general(url):

    r = requests.get(url, timeout=30)
    r.raise_for_status()

    try:
        texto = r.content.decode("utf-8")
    except UnicodeDecodeError:
        texto = r.content.decode("latin-1")
    return json.loads(texto) # guardamos los datos en una lista de diccionarios


def llamada_datos(endpoint, metadatos=False):
    control = llamada_control(endpoint)
    url1, url2 = control["datos"], control["metadatos"]
    
    if metadatos == True:
        return llamada_general(url1), llamada_general(url2)["campos"]

    return llamada_general(url1)


def llamada_metadatos(endpoint):

    url = llamada_control(endpoint)["metadatos"] 

    return llamada_general(url)["campos"]


def descargar_datos_lluvia_actual(estaciones, reintentos=3, espera=60):

    datos_lluvia = []
    fallidas = [] # (estación, motivo) en vez de solo imprimir por pantalla
    metadatos = llamada_metadatos("/observacion/convencional/todas")

    for est in estaciones:
        endpoint = f"/observacion/convencional/datos/estacion/{est}"

        for intento in range(reintentos):
            try:
                datos_lluvia.extend(llamada_datos(endpoint))
                break # en caso de no haber problemas salimos del bucle de reintentos

            except HTTPError as e: # para acortar 
                if e.response.status_code == 429: # limite de llamadas ha sido alcanzado: toca esperar
                    time.sleep(espera)
                    continue
                fallidas.append((est, e.response.status_code))
                break

            except RuntimeError as e:
                if e.args[0] == 429:          # límite avisado dentro del JSON
                    time.sleep(espera)
                    continue
                fallidas.append((est, e.args))
                break

            except RequestException: # Claude Opus 5.5 (bajo)
                time.sleep(5)

        else:
            fallidas.append((est, "reintentos agotados"))



    return datos_lluvia, metadatos, fallidas


**Estaciones con sus `idema` correspondientes**:
- filtraremos para coger las de la provincia de Valencia
- establecemos un criterio para seleccionar las estaciones relevantes para el proyecto
- *criterio*: Estaciones que se encuentren a un máximo de 30km del centro de Valencia

In [ ]:
endpoint = "/valores/climatologicos/inventarioestaciones/todasestaciones"

datos = llamada_datos(endpoint) # cargamos los datos de todas las estaciones en una lista de diccionarios
estaciones = {} # vamos a crear nuestro propio diccionario con las estaciones y sus idema
for i in range(len(datos)):
    if datos[i]["provincia"] == "VALENCIA": # queremos las estaciones de Valencia
        estaciones[datos[i]["indicativo"]] = datos[i]["nombre"]

print(estaciones)

"""
seleccionamos las estaciones que cumplen con el criterio que hemos establecido:
Valencia, UPV, Viveros, Aeropuerto, Llíria y Sagunt
"""

estaciones = {'8409X': 'LLÍRIA',
              '8414A': 'VALENCIA AEROPUERTO',
              '8416': 'VALÈNCIA',
              '8416X': 'VALENCIA, UPV',
              '8416Y': 'VALÈNCIA, VIVEROS',
              '8446Y': 'SAGUNT/SAGUNTO'
}

**Datos de LLuvia**:
- cogeremos los datos de las estaciones en la provincia de Valencia (que posean datos meteorológicos) utilizando los idema extraídos anteriormente
- datos de las últimas 12 horas
- el endpoint contiene la ruta y el parámetro idema (identificador de la estación)
- usamos `.extend()`en lugar de `.append()` para aplanar 
- datos_lluvia es una lista de diccionarios donde cada diccionario corresponde a la medición en 1 hora de una estación particular. 
- Hay estaciones que pueden no ofrecer datos en vivo (UPV, VIVEROS), estas se *guardan en una lista con su motivo* 


In [ ]:
datos_lluvia, metadatos, fallidas = descargar_datos_lluvia_actual(estaciones)

**Comprobaciones iniciales**:
- `len(datos_lluvia)`corresponde a la cantidad total de observaciones individuales que hemos recibido
-  la primera hora observada de cada una de las estaciones


In [ ]:
print(len(datos_lluvia))
print(datos_lluvia[0:-1:12])

**Investigar datos de Lluvia**:
- Convertir la lista de diccionarios a un dataframe
- Sacar la información del dataframe y datos básicos sobre su contenido

In [ ]:
df_lluvia = pd.DataFrame(datos_lluvia) # cargamos los datos en un dataframe que será lo que manejaremos

In [ ]:
df_lluvia.info() # contenido del dataframe y el tipo de dato de las variables

In [ ]:
df_lluvia.describe().T # estadísticos de los datos

**Procesar la hora**:
- tenemos que convertir la columna `fint`a hora local y en un formato que podamos interpretar
- crearemos una nueva columna `hora` que almacena la hora local 

In [ ]:
df_lluvia["fint"] = pd.to_datetime(df_lluvia["fint"], utc=True) # Claude Opus 5.5 (bajo)

df_lluvia["hora"] = df_lluvia["fint"].dt.tz_convert("Europe/Madrid") # Claude Opus 5.5 (bajo) 

In [ ]:
df_lluvia

**EXTRACCIÓN DE METADATOS PARA LA COMPRENSIÓN DE LAS VARIABLES INVOLUCRADAS**
- los metadatos se extraen usando la URL cotenida en el valor de la clave "metadatos" en la llamada de control
- los metadatos incluyen información sobre las variables y su contenido
- utilizaremos esta información para seleccionar las variables relevantes para nuestro proyecto 

*la próxima celda incluirá la definicón de la función que utilizaremos para extraer los metadatos y un ejemplo con los datos meteorlógicos*

*Incluiré la función en la celda inicial, por tanto, la ejecución de esta celda es opcional* 

**LLAMADA_METADATOS():**
- replica lo que hace la llamda general, pero es específica de los metadatos, de esta forma tenemos que:
    - llamada_general extrae información a cerca del endpoint
    - llamada_datos extrae los datos para un endpoint concreto
    - llamada_metadatos extraer la información sobre los datos extraídos (sus variables etc.)
- recibe un endpoint
- devuelve un diccionario que incluye información a cerca del origen de los datos y sus condiciones de uso, además de (y el motivo de esta llamada), la información de los distintos campos (variables), está guardado como una lista de diccionarios dentro de la clave "campos" del diccionario exterior. 

In [ ]:

## Ejmeplo:

endpoint = "/valores/climatologicos/inventarioestaciones/todasestaciones" 

# print(llamada_metadatos(endpoint)) # esto devuelve todos los metadatos (descomentar para comprobar)

print(llamada_metadatos(endpoint)) # esto devuelve solo la lista de diccionarios con la información de los campos


De cara a la versión final del las funciones, como según la API de OpenData de la AEMET la API tiene un límite de 40 llamadas/minuto por usuario, y además cada URL proporcionada por la llamada control tiene una caducidad de 5 minutos, para prevenir la caducidad vamos a fusionar ambas llamadas en una sola y el usuario será capaz de seleccionar si quiere los metadatos o no usando una parámetro (True/False) en la llamada. 

Los cambios pertinentes ya se han realizado sobre el código por lo que la celda previa solo sirve un rol de exploración
También vamos a mantener la función llamada_metadatos para utilizarla en el análisis de las variables de datos_lluvia.

In [ ]:
df_metadatos = pd.DataFrame(metadatos,).set_index("id")

df_metadatos = df_metadatos.loc[df_metadatos.index.isin(df_lluvia)]

df_metadatos

**ANALISIS DE LOS METADATOS CLIMATOLÓGICOS**:

Éstas son las variables/campos con los que vamos a trabajar: 

- idema: indicativo climatológico de la estación, string
- ubi: nombre de la estación, string
- fint: Fecha hora final del período de observación, se trata de datos del periodo de la hora anterior a la indicada por este campo (hora UTC), string
- prec: Precipitación acumulada, medida por el pluviómetro, durante los 60 minutos anteriores a la hora indicada por el período de observación, float (l/m2)
- vmax: Velocidad máxima del viento, valor máximo del viento mantenido 3 segundos y registrado en los 60 minutos anteriores a la hora indicada por el período de observación (m/s), float - (ya ha pasado)
- vv: Velocidad media del viento, media escalar de las muestras adquiridas cada 0,25 ó 1 segundo en el período de 10 minutos anterior al indicado (m/s), float  - (está pasando)
- ta: Temperatura instantánea del aire correspondiente a la fecha dada (Celcius), float


Las demás variables las descartamos por no estar dentro del alcance del proyecto, no aportar información útil, limitaciones en el número de estaciones que registran valores para esas variables, o por ser redundantes con otras variables. 

**MEJORA EN LA DESCARGA DE DATOS METEOROLÓGICOS:**

Hasta ahora solo nos hemos dedicado a descargar los datos que necesitabamos para nuestro generador de excusas, pero habíamos ignorado el manejo de errores y excepciones. 
Ahora vamos a mejorar el diseño de la función de descarga de datos para solventar esto

**GUARDAR EL BRUTO Y COMPROBARLO:**
- El nombre lleva la hora de descarga en UTC, así que cada ejecución crea un archivo nuevo y no sobrescribe el anterior. La Z indica que es UTC, igual que fint.
- encoding="utf-8" y ensure_ascii=False hacen que "LLÍRIA" o "VALÈNCIA" se guarden como texto legible y no como \u00cd.
- indent=2 solo sirve para que el archivo se pueda leer si lo abres a mano.
- El assert comprueba que se puede leer lo que has escrito. Si no coincide, se para y te avisa.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

CARPETA = Path("data")
CARPETA.mkdir(exist_ok=True)                      # crea la carpeta si no existe

momento_descarga = datetime.now(timezone.utc)
marca = momento_descarga.strftime("%Y%m%dT%H%MZ") 
ruta = CARPETA / f"aemet_obs_{marca}.json"

with open(ruta, "w", encoding="utf-8") as f:
    json.dump(datos_lluvia, f, ensure_ascii=False, indent=2)

# comprobación: lo que se lee es igual a lo que se ha guardado Claude Opus 5.5 (bajo)
with open(ruta, encoding="utf-8") as f:
    recargado = json.load(f)
assert recargado == datos_lluvia, "El archivo no coincide con los datos descargados"
print(f"Guardado: {ruta} ({len(recargado)} registros)")

**LIMPIEZA**:
- reindex(columns=...) se queda con tus variables. Si en alguna descarga falta una columna (por ejemplo, ninguna estación ha medido viento), la crea vacía en vez de dar error, cosa que sí pasaría si seleccionaras las columnas con corchetes.
- drop_duplicates en (idema, fint) elimina las repeticiones si las hay. Si las 13 observaciones eran horas distintas, no borra nada.
- antiguedad_h indica cuántas horas han pasado desde cada observación, y la usarás para el umbral de frescura.
- El parámetro ahora es para la demo reproducible. Si limpias un archivo antiguo, pasas el momento en que se descargó y no la hora actual. Si no, todos los datos de un archivo de ayer aparecerían como caducados.
- Los NaN se quedan como están. Las filas de Sagunt no se borran porque siguen valiendo para la lluvia y la temperatura. En el bloque 3 verás por qué no molestan.

In [ ]:
COLUMNAS = ["idema", "ubi", "fint", "prec", "vmax", "vv", "ta"]

def limpiar(datos, ahora=None):

    df = pd.DataFrame(datos).reindex(columns=COLUMNAS) # Claude Opus 5.5 (bajo)
    df["fint"] = pd.to_datetime(df["fint"], utc=True)
    df["hora"] = df["fint"].dt.tz_convert("Europe/Madrid")
    df = df.drop_duplicates(subset=["idema", "fint"])

    ahora = ahora if ahora is not None else pd.Timestamp.now(tz="UTC")
    df["antiguedad_h"] = (ahora - df["fint"]).dt.total_seconds() / 3600

    return df.sort_values(["idema", "fint"]).reset_index(drop=True)

df = limpiar(datos_lluvia)